In [1]:
import csv
from collections import defaultdict
from typing import Dict, List, Tuple

CSV_PATH = "ucl_2425_league_phase_results.csv"


#load match data
def load_matches(csv_path: str) -> List[dict]:
    matches = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            matches.append({
                "matchday": int(row["matchday"]),
                "home":     row["home"],
                "away":     row["away"],
                "hg":       int(row["home_goals"]),
                "ag":       int(row["away_goals"]),
            })
    return matches


#standard points (baseline)
def outcome_points(goals_for: int, goals_against: int) -> int:
    if goals_for > goals_against: return 3
    if goals_for == goals_against: return 1
    return 0


def compute_standard_points(matches: List[dict]) -> Dict[str, int]:
    points = defaultdict(int)
    for m in matches:
        h, a   = m["home"], m["away"]
        hg, ag = m["hg"],   m["ag"]
        points[h] += outcome_points(hg, ag)
        points[a] += outcome_points(ag, hg)
    return dict(points)


#step 1: Build weight matrix W
def build_credit_weights(
    matches: List[dict],
    win_weight: float = 3.0,
    draw_weight: float = 1.0,
) -> Tuple[List[str], Dict[str, int], List[List[float]]]:
    team_set = set()
    for m in matches:
        team_set.add(m["home"])
        team_set.add(m["away"])

    teams = sorted(team_set)
    idx   = {t: k for k, t in enumerate(teams)}
    n     = len(teams)
    W     = [[0.0] * n for _ in range(n)]

    for m in matches:
        h, a   = m["home"], m["away"]
        hg, ag = m["hg"],   m["ag"]
        hi, ai = idx[h],    idx[a]

        if hg > ag:
            W[hi][ai] += win_weight
        elif ag > hg:
            W[ai][hi] += win_weight
        else:
            W[hi][ai] += draw_weight
            W[ai][hi] += draw_weight

    return teams, idx, W



#step 1b: Reversed weight matrix
def build_credit_weights_reversed(
    matches: List[dict],
    win_weight: float = 3.0,
    draw_weight: float = 1.0,
) -> Tuple[List[str], Dict[str, int], List[List[float]]]:
    team_set = set()
    for m in matches:
        team_set.add(m["home"])
        team_set.add(m["away"])

    teams = sorted(team_set)
    idx   = {t: k for k, t in enumerate(teams)}
    n     = len(teams)
    W     = [[0.0] * n for _ in range(n)]

    for m in matches:
        h, a   = m["home"], m["away"]
        hg, ag = m["hg"],   m["ag"]
        hi, ai = idx[h],    idx[a]

        if hg > ag:
            W[ai][hi] += win_weight   # reversed: loser gets credit
        elif ag > hg:
            W[hi][ai] += win_weight
        else:
            W[ai][hi] += draw_weight
            W[hi][ai] += draw_weight

    return teams, idx, W


#step 2: Column-stochastic transition matrix
def normalise_to_transition_matrix(
    W: List[List[float]],
) -> List[List[float]]:
    n = len(W)
    P = [[0.0] * n for _ in range(n)]

    for j in range(n):
        col_sum = sum(W[i][j] for i in range(n))
        if col_sum > 0.0:
            for i in range(n):
                P[i][j] = W[i][j] / col_sum
        else:
            uniform = 1.0 / n
            for i in range(n):
                P[i][j] = uniform

    return P


#step 3: Teleportation
def apply_teleportation(
    P: List[List[float]],
    alpha: float = 0.7,
) -> List[List[float]]:
    n       = len(P)
    P_tilde = [[0.0] * n for _ in range(n)]
    base    = (1.0 - alpha) / n

    for j in range(n):
        for i in range(n):
            P_tilde[i][j] = alpha * P[i][j] + base

    return P_tilde

#step 4: Stationary distribution via power iteration

def power_iteration_stationary(
    P_tilde: List[List[float]],
    tol: float = 1e-12,
    max_iters: int = 10000,
) -> List[float]:
    n = len(P_tilde)
    p = [1.0 / n] * n

    for _ in range(max_iters):
        p_next = [0.0] * n
        for i in range(n):
            for j in range(n):
                p_next[i] += P_tilde[i][j] * p[j]

        diff = sum(abs(p_next[i] - p[i]) for i in range(n))
        p    = p_next

        if diff < tol:
            break

    total = sum(p)
    if total > 0:
        p = [x / total for x in p]

    return p


#ranking and analysis utilities
def get_rank(scored: Dict[str, float]) -> Dict[str, int]:
    rows = sorted(scored.items(), key=lambda x: (-x[1], x[0]))
    return {team: r + 1 for r, (team, _) in enumerate(rows)}


def kendall_tau(rank_a: Dict[str, int], rank_b: Dict[str, int]) -> float:
    teams = list(rank_a.keys())
    n = len(teams)
    c = d = 0
    for p in range(n):
        for q in range(p + 1, n):
            t1, t2 = teams[p], teams[q]
            prod = (rank_a[t1] - rank_a[t2]) * (rank_b[t1] - rank_b[t2])
            if prod > 0:   c += 1
            elif prod < 0: d += 1
    return (c - d) / (n * (n - 1) // 2)


def spearman_rho(rank_a: Dict[str, int], rank_b: Dict[str, int]) -> float:
    teams = list(rank_a.keys())
    n = len(teams)
    d_sq = sum((rank_a[t] - rank_b[t]) ** 2 for t in teams)
    return 1.0 - (6.0 * d_sq) / (n * (n * n - 1))


def topk_overlap(rank_a: Dict[str, int], rank_b: Dict[str, int], k: int) -> int:
    top_a = {t for t, r in rank_a.items() if r <= k}
    top_b = {t for t, r in rank_b.items() if r <= k}
    return len(top_a & top_b)


#main
def main():
    matches = load_matches(CSV_PATH)
    points  = compute_standard_points(matches)
    std_rank = get_rank({t: float(v) for t, v in points.items()})

    alpha = 0.7

    #original pi
    teams_o, idx_o, W_o = build_credit_weights(matches)
    P_o                  = normalise_to_transition_matrix(W_o)
    P_tilde_o            = apply_teleportation(P_o, alpha=alpha)
    pi_o                 = power_iteration_stationary(P_tilde_o)
    pi_o_by_team         = {teams_o[i]: pi_o[i] for i in range(len(teams_o))}

    #reversed pi
    teams_r, idx_r, W_r = build_credit_weights_reversed(matches)
    P_r                  = normalise_to_transition_matrix(W_r)
    P_tilde_r            = apply_teleportation(P_r, alpha=alpha)
    pi_r                 = power_iteration_stationary(P_tilde_r)
    pi_r_by_team         = {teams_r[i]: pi_r[i] for i in range(len(teams_r))}

    #final score: pi_original - pi_reversed
    teams      = sorted(set(teams_o) | set(teams_r))
    score      = {t: pi_o_by_team.get(t, 0.0) - pi_r_by_team.get(t, 0.0) for t in teams}
    score_rank = get_rank(score)
    score_sorted = sorted(score.items(), key=lambda x: (-x[1], x[0]))

    #print ranking table
    print(f"Teleportation alpha = {alpha}")
    print(f"{'Pos':>3}  {'Team':25}  {'Pts':>4}  {'pi_o - pi_r':>12}  {'Delta':>6}")
    print("=" * 62)
    for pos, (team, s) in enumerate(score_sorted, start=1):
        pts   = points.get(team, 0)
        delta = std_rank.get(team, pos) - score_rank.get(team, pos)
        print(f"{pos:>3}  {team:25}  {pts:>4}  {s:>12.8f}  {delta:>+6}")
    print()

    #rank correlation vs standard points table
    tau = kendall_tau(std_rank, score_rank)
    rho = spearman_rho(std_rank, score_rank)

    print("=" * 50)
    print("Rank correlation vs standard points table")
    print("=" * 50)
    print(f"  Kendall's tau:  {tau:+.4f}")
    print(f"  Spearman's rho: {rho:+.4f}")
    print()

    #top-k overlap
    print("Top-k overlap with standard points table")
    print(f"  Top  8 (auto qualification): {topk_overlap(std_rank, score_rank,  8)} / 8")
    print(f"  Top 24 (play-off cut):       {topk_overlap(std_rank, score_rank, 24)} / 24")
    print()


if __name__ == "__main__":
    main()

Teleportation alpha = 0.7
Pos  Team                        Pts   pi_o - pi_r   Delta
  1  Liverpool                    21    0.04038773      +0
  2  Bayer Leverkusen             16    0.03271457      +5
  3  PSV Eindhoven                14    0.03164409     +11
  4  Inter Milan                  19    0.02999421      +0
  5  Barcelona                    19    0.02720644      -2
  6  Arsenal                      19    0.02637043      -4
  7  Monaco                       13    0.02451967     +11
  8  Real Madrid                  15    0.01864782      +5
  9  Lille                        16    0.01794820      -1
 10  Atalanta                     15    0.01586375      -1
 11  Atlético Madrid              18    0.01449681      -6
 12  Benfica                      13    0.01313637      +3
 13  Juventus                     12    0.01197101      +8
 14  Brest                        13    0.00774431      +2
 15  Milan                        15    0.00719588      -3
 16  Aston Villa              